# Notebook 3 — Profile-aware Deployment

Reads the run + `testbed_config.yaml` written by the dashboard, detects the
deployment profile (`cots` or `rfsim`) automatically, and emits the right
deployment package. No manual edits.


## Cell 1 — Config & UE mapping (generic)

Detects `KIND` from `testbed_config.yaml`, builds a generic UE endpoint model,
and runs only the checks that apply to that profile.


In [1]:
import json, shutil, warnings
from collections import defaultdict, Counter
from datetime import datetime
from pathlib import Path
from typing import Dict, List
import pandas as pd, yaml
warnings.filterwarnings("ignore")

def find_project_root() -> Path:
    cur = Path.cwd()
    for cand in [cur, *cur.parents]:
        if (cand / "data").is_dir() and (cand / "artifacts").is_dir():
            return cand
    raise FileNotFoundError("Could not find project root.")

PROJECT_ROOT = find_project_root()
TRAFFIC_PROFILES_DIR = PROJECT_ROOT / "traffic_profiles"

available_runs = sorted(d for d in TRAFFIC_PROFILES_DIR.iterdir()
                        if d.is_dir() and d.name.startswith("run_")
                        and (d / "mgen_scripts").exists())
if not available_runs:
    raise FileNotFoundError("No MGEN runs found. Run Notebook 2 first.")
RUN_DIR = available_runs[-1]
MGEN_SCRIPTS_DIR = RUN_DIR / "mgen_scripts"

# ── load testbed config ───────────────────────────────────────────────────────
_tb_path = PROJECT_ROOT / "testbed_config.yaml"
if not _tb_path.exists():
    raise FileNotFoundError(f"testbed_config.yaml not found at {_tb_path}")
with open(_tb_path) as _f:
    _tb = yaml.safe_load(_f)

# ── detect deployment KIND (the profile adapter) ──────────────────────────────
# Primary signal: the `testbed` label. Structural fallback: boxes carrying a
# `container` field are containerised (rfsim); otherwise physical hosts (cots).
_label = str(_tb.get("testbed", "")).lower()
_boxes = (_tb.get("ues") or {}).get("boxes") or {}
if "rfsim" in _label:
    KIND = "rfsim"
elif "cots" in _label:
    KIND = "cots"
else:
    KIND = "rfsim" if any("container" in (b or {}) for b in _boxes.values()) else "cots"

ALLOW_PLACEHOLDER_HOSTS = bool((_tb.get("flags") or {}).get("allow_placeholder_hosts", False))
ALLOW_INVALID_RUN       = bool((_tb.get("flags") or {}).get("allow_invalid_run", False))

# ── DN config ─────────────────────────────────────────────────────────────────
_dn = _tb["dn"]
CN5G_SSH_HOST = str(_dn["ssh_host"])
DN_CONFIG = {"container": str(_dn["container"]), "ssh_host": CN5G_SSH_HOST,
             "ip": None, "mgen_dir": str(_dn["mgen_dir"])}

# ── UE host settings + generic endpoint model (both profiles fill this) ───────
_ues = _tb["ues"]
UE_USERNAME      = str(_ues.get("username", ""))
UE_MGEN_BASE_DIR = str(_ues["mgen_dir"])

def _is_placeholder_ip(ip: str) -> bool:
    ip = (ip or "").strip()
    return ip == "" or "x" in ip.lower() or "<" in ip

PHYSICAL_UES: Dict[str, Dict] = {}
for box_name, box in _boxes.items():
    ip = str(box.get("ip", "")).strip()
    PHYSICAL_UES[box_name] = {
        "ssh_host":   str(box["ssh_host"]),
        "ip":         ip,
        "interface":  str(box["interface"]),
        "container":  str(box["container"]) if box.get("container") else None,
        "dynamic_ip": (KIND == "rfsim") or _is_placeholder_ip(ip),
    }

UE_NAME_MAP: Dict[str, str] = {str(g): str(p) for g, p in (_tb.get("ue_name_map") or {}).items()}

print(f"  Testbed config : {_tb_path.name}")
print(f"  Testbed label  : {_tb.get('testbed', 'unspecified')}")
print(f"  Deployment KIND: {KIND}")

# ── read run config ───────────────────────────────────────────────────────────
with open(RUN_DIR / "config.json") as f:
    _cfg = json.load(f)
DN_CONFIG["ip"]     = _cfg["network"]["dn_ip"]
DL_PORT             = _cfg["network"]["dl_port"]
UL_PORT             = _cfg["network"]["ul_port"]
SIMULATION_DURATION = _cfg["simulation_duration"]
APPS                = _cfg["apps"]

# ── NB1 validation gate ───────────────────────────────────────────────────────
_v = _cfg.get("validation", {})
if not _v.get("passed", True) and not ALLOW_INVALID_RUN:
    raise ValueError(f"Run '{RUN_DIR.name}' failed Notebook 1 validation. "
                     "Set allow_invalid_run: true to override.")

# ── manifest + ue_name_map validation (common to both profiles) ───────────────
manifest_df     = pd.read_csv(MGEN_SCRIPTS_DIR / "manifest.csv")
generated_names = set(manifest_df["ue_name"].tolist())

unmapped = generated_names - set(UE_NAME_MAP)
if unmapped:
    raise ValueError(f"ue_name_map incomplete. Missing: {sorted(unmapped)}")
phantom = set(UE_NAME_MAP) - generated_names
if phantom:
    raise ValueError(f"ue_name_map references unknown generated UEs: {sorted(phantom)}")
unknown_box = set(UE_NAME_MAP.values()) - set(PHYSICAL_UES)
if unknown_box:
    raise ValueError(f"ue_name_map references unknown boxes: {sorted(unknown_box)}")

# ── COTS-ONLY topology checks (skip for rfsim: containers share one node) ─────
if KIND == "cots":
    _multi = {b: n for b, n in Counter(UE_NAME_MAP.values()).items() if n > 1}
    if _multi:
        raise ValueError(f"Multiple generated UEs mapped to one physical box "
                         f"is unsupported on COTS. Conflicts: {_multi}")
    _used_ips = [PHYSICAL_UES[b]["ip"] for b in set(UE_NAME_MAP.values())]
    _dups = sorted({ip for ip in _used_ips if _used_ips.count(ip) > 1})
    if _dups:
        raise ValueError(f"Duplicate physical UE IP(s): {_dups}")

# ── host placeholder check (both) ─────────────────────────────────────────────
_used_boxes = set(UE_NAME_MAP.values())
_hosts = [CN5G_SSH_HOST] + [PHYSICAL_UES[b]["ssh_host"] for b in _used_boxes]
_bad = [h for h in _hosts if "<experiment>" in h]
if _bad and not ALLOW_PLACEHOLDER_HOSTS:
    raise ValueError("Replace <experiment> in SSH hostnames:\n" + "\n".join(_bad))

# ── build per-generated-UE mapping (keys kept compatible with the COTS path) ──
ue_mapping: List[Dict] = []
for gen_name, box_name in UE_NAME_MAP.items():
    row = manifest_df[manifest_df["ue_name"] == gen_name].iloc[0]
    box = PHYSICAL_UES[box_name]
    ue_mapping.append({
        "generated_ue_name":     gen_name,
        "generated_ue_ip":       row["ue_ip"],
        "physical_ue_name":      box_name,
        "physical_ue_ssh_host":  box["ssh_host"],
        "physical_ue_ip":        box["ip"],
        "physical_ue_interface": box["interface"],
        "container":             box["container"],
        "dynamic_ip":            box["dynamic_ip"],
        "dl_rx_script":          row["dl_rx_script"],
        "ul_tx_script":          row["ul_tx_script"],
        "ue_class":              row["ue_class"],
        "n_dl_events":           row["n_dl_events"],
        "n_ul_events":           row["n_ul_events"],
    })
box_to_ues: Dict[str, List[Dict]] = defaultdict(list)
for m in ue_mapping:
    box_to_ues[m["physical_ue_name"]].append(m)

print(f"  Run            : {RUN_DIR.name}")
print(f"  Apps           : {', '.join(APPS)}  |  duration {SIMULATION_DURATION}s")
print(f"  DN             : {DN_CONFIG['container']} @ {CN5G_SSH_HOST}  (DN IP {DN_CONFIG['ip']})")
print(f"  UEs            : {len(ue_mapping)} generated")
for m in ue_mapping:
    tgt = f"{m['container']} [dynamic]" if m["dynamic_ip"] else f"{m['physical_ue_ip']}"
    print(f"    {m['generated_ue_name']} ({m['ue_class']}) -> {m['physical_ue_name']} -> {tgt}  iface {m['physical_ue_interface']}")
print(f"  KIND={KIND}: Cell 2 will emit the {'RFsim live-resolve runner' if KIND=='rfsim' else 'COTS static guide'}")


  Testbed config : testbed_config.yaml
  Testbed label  : powder_rfsim_docker
  Deployment KIND: rfsim
  Run            : run_filimo_igap_youtube_telegram_aparat_20260626_105335
  Apps           : filimo, igap, youtube, telegram, aparat  |  duration 200.0s
  DN             : rfsim5g-oai-ext-dn @ ghinwa@pc855.emulab.net  (DN IP 192.168.72.135)
  UEs            : 3 generated
    ue1 (heavy) -> ue1 -> rfsim5g-oai-nr-ue1 [dynamic]  iface oaitun_ue1
    ue2 (medium) -> ue2 -> rfsim5g-oai-nr-ue2 [dynamic]  iface oaitun_ue1
    ue3 (light) -> ue3 -> rfsim5g-oai-nr-ue3 [dynamic]  iface oaitun_ue1
  KIND=rfsim: Cell 2 will emit the RFsim live-resolve runner


## Cell 2 — Generate deployment package

`cots` -> static guide. `rfsim` -> `deploy_rfsim.sh` that resolves live PDU IPs.


In [2]:
# Cell 2 — generate the deployment package for the detected KIND.
#   cots  -> static IP rewrite + per-box manual guide (unchanged behaviour)
#   rfsim -> emit a live-resolve runner (PDU IPs resolved at deploy time)

if KIND == "cots":
    # ── output directories ────────────────────────────────────────────────────────

    deployment_dir      = RUN_DIR / "deployment"
    updated_scripts_dir = deployment_dir / "updated_scripts"
    deployment_dir.mkdir(exist_ok=True)
    updated_scripts_dir.mkdir(exist_ok=True)

    ts          = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name    = f"mgen_{ts}"
    scripts_abs = updated_scripts_dir.resolve()
    dur_s       = SIMULATION_DURATION
    dur_min     = dur_s / 60

    print("="*72)
    print("  GENERATING DEPLOYMENT FILES")
    print("="*72)
    print(f"\n  Output dir     : "
          f"{deployment_dir.relative_to(PROJECT_ROOT)}")
    print(f"  Remote run name: {run_name}")

    # ══════════════════════════════════════════════════════════════════════════════
    # verify all required scripts exist before writing anything
    # ══════════════════════════════════════════════════════════════════════════════

    print("\n  Pre-flight checks...")

    required = (
        [MGEN_SCRIPTS_DIR / "dn_dl_tx.mgn",
         MGEN_SCRIPTS_DIR / "dn_ul_rx.mgn"]
        + [MGEN_SCRIPTS_DIR / m["dl_rx_script"] for m in ue_mapping]
        + [MGEN_SCRIPTS_DIR / m["ul_tx_script"] for m in ue_mapping]
    )

    missing = [p for p in required if not p.exists()]
    if missing:
        for p in missing:
            print(f"  ❌  Missing: {p.relative_to(PROJECT_ROOT)}")
        raise FileNotFoundError(
            f"{len(missing)} script(s) missing — re-run Notebook 2."
        )

    print(f"  ✅  All {len(required)} required scripts present")

    # ══════════════════════════════════════════════════════════════════════════════
    #  STEP 1 — Rewrite dn_dl_tx.mgn with real UE IPs
    #
    #  Every generated UE is in UE_NAME_MAP (validated in Cell 1), so every
    #  placeholder IP gets replaced. No placeholder IPs remain after this step.
    # ══════════════════════════════════════════════════════════════════════════════

    print("\n  [1/4] Rewriting dn_dl_tx.mgn with real UE IPs...")

    with open(MGEN_SCRIPTS_DIR / "dn_dl_tx.mgn") as fh:
        dn_dl_content = fh.read()

    for m in ue_mapping:
        old = m["generated_ue_ip"]
        new = m["physical_ue_ip"]
        dn_dl_content = dn_dl_content.replace(f"DST {old}/", f"DST {new}/")
        print(f"    {old} → {new}")

    # Verify no placeholder IPs remain

    all_generated_ips = {m["generated_ue_ip"] for m in ue_mapping}
    residual = [ip for ip in all_generated_ips if f"DST {ip}/" in dn_dl_content]
    if residual:
        raise RuntimeError(
            f"Placeholder IPs still in dn_dl_tx.mgn after rewrite: {residual}\n"
            f"This should not happen — check UE_NAME_MAP."
        )

    (updated_scripts_dir / "dn_dl_tx.mgn").write_text(dn_dl_content)
    shutil.copy(MGEN_SCRIPTS_DIR / "dn_ul_rx.mgn",
                updated_scripts_dir / "dn_ul_rx.mgn")

    for m in ue_mapping:
        for script in [m["dl_rx_script"], m["ul_tx_script"]]:
            shutil.copy(MGEN_SCRIPTS_DIR / script,
                        updated_scripts_dir / script)

    n_scripts = len(list(updated_scripts_dir.glob("*.mgn")))
    print(f"  ✅  {n_scripts} scripts in updated_scripts/  "
          f"(no placeholder IPs remain)")

    # ══════════════════════════════════════════════════════════════════════════════
    #  STEP 2 — Build command strings
    #
    #  Commands are built per generated UE, then grouped by physical box.
    #  When multiple generated UEs share a box, the command file lists all
    #  their scripts — the operator runs each in its own terminal.
    #
    #  docker exec: no -it flag — all calls are non-interactive scripted
    #  commands; -it causes tty errors through scripted SSH.
    #
    #  Paths: absolute throughout — tilde expansion is unreliable in
    #  quoted ssh/scp commands across environments.
    # ══════════════════════════════════════════════════════════════════════════════

    print("\n  [2/4] Building deployment commands...")

    cn5g   = DN_CONFIG["ssh_host"]
    cname  = DN_CONFIG["container"]
    dn_run = f"{DN_CONFIG['mgen_dir']}/{run_name}"

    # ── DN command sections (one per section, not per UE) ─────────────────────────

    ue_ips_str = ", ".join(m["physical_ue_ip"] for m in ue_mapping)

    dn_cmds: Dict[str, str] = {}

    dn_cmds["setup"] = f"""\
    # ── DN SETUP        ─────────────────────────────────────────────
    # Docker is on the remote CN5G host.
    # Sequence: scp to CN5G host → docker cp into container.

    # 1. Create directory inside the container (no -it: non-interactive)
    ssh {cn5g} "sudo docker exec {cname} mkdir -p {dn_run}"

    # 2. SCP scripts to CN5G host
    scp "{scripts_abs}/dn_dl_tx.mgn" {cn5g}:/tmp/dn_dl_tx.mgn
    scp "{scripts_abs}/dn_ul_rx.mgn" {cn5g}:/tmp/dn_ul_rx.mgn

    # 3. Copy from CN5G host into container
    ssh {cn5g} "sudo docker cp /tmp/dn_dl_tx.mgn {cname}:{dn_run}/"
    ssh {cn5g} "sudo docker cp /tmp/dn_ul_rx.mgn {cname}:{dn_run}/"

    # 4. Verify (no -it: non-interactive)
    ssh {cn5g} "sudo docker exec {cname} ls -lh {dn_run}"
    """

    dn_cmds["start_receiver"] = f"""\
    # ── DN UPLINK RECEIVER  (start FIRST — before any UE sender) ──────────────────
    # No interface flag needed — DN receives on all interfaces.
    # Sends to: {ue_ips_str}
    ssh {cn5g}
    sudo docker exec {cname} bash -c \\
      'mgen input {dn_run}/dn_ul_rx.mgn output {dn_run}/dn_ul_rx.log'
    # Leave this running.
    """

    dn_cmds["start_sender"] = f"""\
    # ── DN DOWNLINK SENDER  (start AFTER all UE receivers are ready) ──────────────
    # No interface flag needed — DN has one relevant interface.
    ssh {cn5g}
    sudo docker exec {cname} bash -c \\
      'mgen txlog input {dn_run}/dn_dl_tx.mgn output {dn_run}/dn_dl_tx.log'
    # Runs for {dur_s}s ({dur_min:.1f} min) then stops automatically.
    """

    dn_cmds["check_logs"] = f"""\
    # ── DN LOG CHECK ──────────────────────────────────────────────────────────────
    ssh {cn5g}

    # UL received (packets FROM UEs)
    sudo docker exec {cname} bash -c \\
      'grep -c "RECV" {dn_run}/dn_ul_rx.log || echo 0'

    # DL sent (packets TO UEs)
    sudo docker exec {cname} bash -c \\
      'grep -c "SEND" {dn_run}/dn_dl_tx.log || echo 0'
    """

    # ── per-physical-box command sections ────────────────────────────────────────
    # Keyed by physical box name; each value is a dict of section → text.
    # When a box hosts multiple generated UEs, all their scripts are listed.

    box_cmds: Dict[str, Dict[str, str]] = {}

    for phy_name, ues_on_box in box_to_ues.items():
        ssh   = PHYSICAL_UES[phy_name]["ssh_host"]
        iface = PHYSICAL_UES[phy_name]["interface"]
        ue_run = f"{UE_MGEN_BASE_DIR}/{run_name}"

        # ── setup: copy all scripts for this box ──────────────────────────────

        gen_labels = ", ".join(
            f"{u['generated_ue_name']} ({u['ue_class']})" for u in ues_on_box
        )
        setup_lines = [
            f"# ── {phy_name.upper()} SETUP  "
            f"(generated: {gen_labels})",
            f"# Run locally.",
            f"",
            f"# 1. Create run directory (absolute path)",
            f'ssh "{ssh}" "mkdir -p {ue_run}"',
            f"",
            f"# 2. Copy scripts",
        ]
        for u in ues_on_box:
            setup_lines += [
                f'scp "{scripts_abs}/{u["dl_rx_script"]}" "{ssh}:{ue_run}/"',
                f'scp "{scripts_abs}/{u["ul_tx_script"]}" "{ssh}:{ue_run}/"',
            ]
        setup_lines += [
            f"",
            f"# 3. Verify",
            f'ssh "{ssh}" "ls -lh {ue_run}"',
        ]

        # ── start receivers: one command per generated UE ─────────────────────

        rx_lines = [
            f"# ── {phy_name.upper()} DOWNLINK RECEIVERS  "
            f"(start FIRST — before DN sender)",
            f"# ⚠️  interface {iface} is REQUIRED on every command.",
            f"# Without it, MGEN binds to eth0 instead of {iface} "
            f"and receives 0 packets.",
        ]
        if len(ues_on_box) > 1:
            rx_lines.append(
                f"# Each receiver needs its own terminal on this box."
            )
        for u in ues_on_box:
            dl_log = u["dl_rx_script"].replace(".mgn", ".log")
            rx_lines += [
                f"",
                f"# {u['generated_ue_name']} ({u['ue_class']})",
                f'ssh "{ssh}"',
                f"sudo mgen interface {iface} \\\\",
                f"  input  {ue_run}/{u['dl_rx_script']} \\\\",
                f"  output {ue_run}/{dl_log}",
            ]

        # ── start senders: one command per generated UE ───────────────────────

        tx_lines = [
            f"# ── {phy_name.upper()} UPLINK SENDERS  "
            f"(start AFTER DN receiver is ready)",
            f"# ⚠️  interface {iface} is REQUIRED on every command.",
            f"# Without it, traffic routes via eth0 and the DN receives "
            f"0 packets.",
        ]
        if len(ues_on_box) > 1:
            tx_lines.append(
                f"# Each sender needs its own terminal on this box."
            )
        for u in ues_on_box:
            ul_log = u["ul_tx_script"].replace(".mgn", ".log")
            tx_lines += [
                f"",
                f"# {u['generated_ue_name']} ({u['ue_class']})",
                f'ssh "{ssh}"',
                f"sudo mgen txlog interface {iface} \\\\",
                f"  input  {ue_run}/{u['ul_tx_script']} \\\\",
                f"  output {ue_run}/{ul_log}",
                f"# Runs for {dur_s}s ({dur_min:.1f} min) then stops.",
            ]

        # ── check logs: one block per generated UE ────────────────────────────

        log_lines = [f"# ── {phy_name.upper()} LOG CHECK"]
        for u in ues_on_box:
            dl_log = u["dl_rx_script"].replace(".mgn", ".log")
            ul_log = u["ul_tx_script"].replace(".mgn", ".log")
            log_lines += [
                f"",
                f'ssh "{ssh}"',
                f"# {u['generated_ue_name']} — DL received (FROM DN)",
                f"grep -c 'RECV' {ue_run}/{dl_log} || echo 0",
                f"# {u['generated_ue_name']} — UL sent (TO DN)",
                f"grep -c 'SEND' {ue_run}/{ul_log} || echo 0",
            ]

        box_cmds[phy_name] = {
            "setup"          : "\n".join(setup_lines),
            "start_receivers": "\n".join(rx_lines),
            "start_senders"  : "\n".join(tx_lines),
            "check_logs"     : "\n".join(log_lines),
        }

    # ══════════════════════════════════════════════════════════════════════════════
    #  STEP 3 — Write command .txt manual guides
    # ══════════════════════════════════════════════════════════════════════════════

    print("\n  [3/4] Writing command files...")

    def write_cmd_file(path: Path, title: str, sections: Dict[str, str]):
        with open(path, "w") as fh:
            fh.write(f"# {title}\n")
            fh.write(f"# Generated: {ts}\n")
            fh.write("#\n")
            fh.write("# ⚠️  MANUAL COMMAND GUIDE — NOT AN EXECUTABLE SCRIPT\n")
            fh.write("#\n")
            fh.write("# These commands are meant to be copied and run ONE BY ONE in separate\n")
            fh.write("# terminals. Running this file directly with `bash` will NOT work:\n")
            fh.write("# the `ssh host` lines open a session that closes immediately,\n")
            fh.write("# and the following commands execute locally rather than remotely.\n")
            fh.write("#\n")
            fh.write("# Follow the steps in DEPLOYMENT_GUIDE.md for the correct sequence.\n")
            fh.write("\n")
            for text in sections.values():
                fh.write(text + "\n\n")

    write_cmd_file(
        deployment_dir / "dn_commands.txt",
        "DN Deployment Commands",
        dn_cmds,
    )
    print("    ✅  dn_commands.txt")

    for phy_name, cmds in box_cmds.items():
        write_cmd_file(
            deployment_dir / f"{phy_name}_commands.txt",
            f"{phy_name.upper()} Deployment Commands",
            cmds,
        )
        print(f"    ✅  {phy_name}_commands.txt")

    # ══════════════════════════════════════════════════════════════════════════════
    #  STEP 4 — Write DEPLOYMENT_GUIDE.md
    # ══════════════════════════════════════════════════════════════════════════════

    print("\n  [4/4] Writing DEPLOYMENT_GUIDE.md...")

    guide_path = deployment_dir / "DEPLOYMENT_GUIDE.md"

    with open(guide_path, "w") as fh:

        fh.write(f"""# MGEN Multi-UE Bidirectional Traffic — Deployment Guide

    **Generated :** {ts}
    **Run       :** {RUN_DIR.name}
    **Apps      :** {', '.join(APPS)}
    **Duration  :** {dur_s}s ({dur_min:.1f} min)
    **UEs       :** {len(ue_mapping)} generated → {len(box_to_ues)} physical boxes

    ---

    ## Overview

    Bidirectional traffic:
    - **Downlink** DN → UEs on UDP port {DL_PORT}
    - **Uplink**   UEs → DN on UDP port {UL_PORT}

    ### UE profile assignments

    | Physical box | Generated UE | Class | DL events | UL events |
    |---|---|---|---|---|
    """)
        for m in ue_mapping:
            fh.write(
                f"| {m['physical_ue_name'].upper()} "
                f"| {m['generated_ue_name']} "
                f"| {m['ue_class']} "
                f"| {m['n_dl_events']:,} "
                f"| {m['n_ul_events']:,} |\n"
            )

        fh.write(f"""
    ---

    ### Critical: UE interface requirement

    Every UE MGEN command **must** include `interface <ue_interface>`
    (configured per box in `testbed_config.yaml`).
    Without it, traffic uses the management Ethernet interface (eth0)
    and never reaches the 5G core.

    | Direction | Node | Interface flag |
    |---|---|---|
    | DL receiver | UE | ✅ `interface <ue_interface>` required |
    | UL sender   | UE | ✅ `interface <ue_interface>` required |
    | DL sender   | DN | ❌ not needed |
    | UL receiver | DN | ❌ not needed |

    ### Critical: remote Docker copy

    Docker runs on the CN5G host, not locally.
    Always: `scp` to CN5G host → `docker cp` into container remotely.

    ### Note: `docker exec` flags

    All `docker exec` calls omit `-it`. Those flags are for interactive
    terminals and cause tty errors when used through scripted SSH.

    ---

    ## Step 1 — Copy scripts to all machines

    Run all commands **locally**.

    ### DN

    ```bash
    {dn_cmds['setup']}
    ```

    ### UEs
    """)

        for phy_name, cmds in box_cmds.items():
            fh.write(f"\n#### {phy_name.upper()}\n\n```bash\n")
            fh.write(cmds["setup"])
            fh.write("\n```\n")

        fh.write(f"""
    ---

    ## Step 2 — Start all receivers (FIRST)

    Open a **separate terminal** for each receiver.
    All receivers must be running before any sender starts.

    ### DN uplink receiver

    ```bash
    {dn_cmds['start_receiver']}
    ```

    ### UE downlink receivers
    """)

        for phy_name, cmds in box_cmds.items():
            fh.write(f"\n#### {phy_name.upper()}\n\n```bash\n")
            fh.write(cmds["start_receivers"])
            fh.write("\n```\n")

        fh.write(f"""
    ---

    ## Step 3 — Wait 5–10 seconds

    Let all receivers stabilise before starting senders.

    ---

    ## Step 4 — Start all senders (SECOND)

    Open a **separate terminal** for each sender.

    ### DN downlink sender

    ```bash
    {dn_cmds['start_sender']}
    ```

    ### UE uplink senders
    """)

        for phy_name, cmds in box_cmds.items():
            fh.write(f"\n#### {phy_name.upper()}\n\n```bash\n")
            fh.write(cmds["start_senders"])
            fh.write("\n```\n")

        fh.write(f"""
    ---

    ## Step 5 — Monitor

    Traffic runs for **{dur_s}s ({dur_min:.1f} min)**.
    Senders stop automatically. Stop receivers with `Ctrl+C` after.

    ---

    ## Step 6 — Check results

    ### DN

    ```bash
    {dn_cmds['check_logs']}
    ```

    ### UEs
    """)

        for phy_name, cmds in box_cmds.items():
            fh.write(f"\n#### {phy_name.upper()}\n\n```bash\n")
            fh.write(cmds["check_logs"])
            fh.write("\n```\n")

        fh.write("""
    ---

    ## Troubleshooting

    | Symptom | Likely cause | Fix |
    |---|---|---|
    | UE RECV = 0 (DL) | Missing `interface <ue_interface>` on receiver | Add `interface <ue_interface>` (from testbed_config.yaml) |
    | DN RECV = 0 (UL) | UE sent via management interface | Add `interface <ue_interface>` to UE sender |
    | SSH refused | Wrong FQDN (missing `-cots-ue`) | Run `ssh <host> hostname` to verify |
    | `docker cp` fails from ... | Docker daemon is remote | SCP to CN5G host first, then `docker cp` |
    | tty error on `docker exec` | `-it` in scripted context | Already fixed — no `-it` in these scripts |
    | Traffic stops early | Last burst < SIMULATION_DURATION | Increase duration in Notebook 1 Cell 2 |
    | Wrong class on wrong box | `UE_NAME_MAP` stale | Update `UE_NAME_MAP` in Notebook 3 Cell 1 |
    | DN sends to placeholder IP | `UE_NAME_MAP` incomplete | Cell 1 now raises an error — fix the map |

    ---

    ## Files reference

    - `DEPLOYMENT_GUIDE.md` — this file
    - `dn_commands.txt` — all DN commands
    """)
        for phy_name in box_cmds:
            fh.write(
                f"- `{phy_name}_commands.txt` — {phy_name.upper()} commands\n"
            )
        fh.write("- `updated_scripts/` — .mgn files with real UE IPs\n")

    print("    ✅  DEPLOYMENT_GUIDE.md")

    # ── final summary ─────────────────────────────────────────────────────────────

    print(f"""
    {'='*72}
      DEPLOYMENT PACKAGE READY
    {'='*72}

      Location : {deployment_dir.relative_to(PROJECT_ROOT)}
      Start    : open DEPLOYMENT_GUIDE.md

      Contents:
    """)
    for item in sorted(deployment_dir.iterdir()):
        if item.is_file():
            print(f"    {item.name}")
        elif item.is_dir():
            count = len(list(item.glob("*.mgn")))
            print(f"    {item.name}/   ({count} .mgn scripts)")

    print()
    print("  ✅  Cell 2 complete")

elif KIND == "rfsim":
    # ===== RFsim Docker deployment package (emitted runner; single source of truth) =====
    deployment_dir      = RUN_DIR / "deployment"
    updated_scripts_dir = deployment_dir / "updated_scripts"
    deployment_dir.mkdir(exist_ok=True)
    updated_scripts_dir.mkdir(exist_ok=True)
    ts       = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = f"mgen_{ts}"

    # preflight
    required = ([MGEN_SCRIPTS_DIR / "dn_dl_tx.mgn", MGEN_SCRIPTS_DIR / "dn_ul_rx.mgn"]
               + [MGEN_SCRIPTS_DIR / m["dl_rx_script"] for m in ue_mapping]
               + [MGEN_SCRIPTS_DIR / m["ul_tx_script"] for m in ue_mapping])
    missing = [p for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError(f"{len(missing)} script(s) missing — re-run Notebook 2: {missing}")
    # copy scripts UNCHANGED — the DN DL IPs are rewritten LIVE by the runner
    for p in required:
        shutil.copy(p, updated_scripts_dir / p.name)

    node         = DN_CONFIG["ssh_host"]
    dn_container = DN_CONFIG["container"]
    dn_ip        = DN_CONFIG["ip"]
    dn_remote    = f"{DN_CONFIG['mgen_dir']}/{run_name}"
    ue_remote    = f"{UE_MGEN_BASE_DIR}/{run_name}"
    subnet       = ".".join(str(dn_ip).split(".")[:3]) + ".0/24"   # data-plane /24 (override if needed)
    scripts_abs  = updated_scripts_dir.resolve()
    dur_s        = SIMULATION_DURATION

    # bash arrays from ue_mapping
    def _arr(name, vals):
        return f"{name}=(" + " ".join(f'"{v}"' for v in vals) + ")"
    ue_names = [m["generated_ue_name"] for m in ue_mapping]
    L = [
        "#!/usr/bin/env bash",
        "# AUTO-GENERATED RFsim deploy runner — one run, idempotent phases.",
        f"# run: {RUN_DIR.name}   node: {node}   generated: {ts}",
        "set -euo pipefail",
        "",
        f'NODE="{node}"',
        f'DN_C="{dn_container}"',
        f'DN_RUN="{dn_remote}"',
        f'UE_RUN="{ue_remote}"',
        f'SUBNET="{subnet}"',
        f'LOCAL="{scripts_abs}"',
        f'DUR={dur_s}',
        "",
        _arr("UES",   ue_names),
        _arr("UE_C",  [m["container"]      for m in ue_mapping]),
        _arr("GENIP", [m["generated_ue_ip"] for m in ue_mapping]),
        _arr("DLRX",  [m["dl_rx_script"]   for m in ue_mapping]),
        _arr("ULTX",  [m["ul_tx_script"]   for m in ue_mapping]),
        "declare -A REALIP",
        "",
        'echo "== 1/6 resolve live UE PDU IPs =="',
        'for i in "${!UES[@]}"; do',
        '  ue="${UES[$i]}"; c="${UE_C[$i]}"',
        '  ip=$(ssh "$NODE" "sudo docker exec $c ip -o -4 addr show oaitun_ue1" | awk "{print \\$4}" | cut -d/ -f1)',
        '  if [ -z "$ip" ]; then echo "  FAIL: $ue ($c) tunnel down — aborting"; exit 1; fi',
        '  REALIP[$ue]="$ip"; echo "  $ue: ${GENIP[$i]} -> $ip"',
        'done',
        "",
        'echo "== 2/6 rewrite DN downlink with live IPs (local) =="',
        'cp "$LOCAL/dn_dl_tx.mgn" /tmp/dn_dl_tx.mgn',
        'for i in "${!UES[@]}"; do',
        '  sed -i.bak "s#DST ${GENIP[$i]}/#DST ${REALIP[${UES[$i]}]}/#g" /tmp/dn_dl_tx.mgn',
        'done',
        'if grep -qE "DST (12\\.1\\.1\\.[0-9]+)/" /tmp/dn_dl_tx.mgn; then :; fi',
        "",
        'echo "== 3/6 re-assert data-plane route (iface flag does NOT force egress on rfsim) =="',
        'for i in "${!UES[@]}"; do',
        '  ssh "$NODE" "sudo docker exec ${UE_C[$i]} ip route replace $SUBNET dev oaitun_ue1"',
        'done',
        "",
        'echo "== 4/6 copy scripts into DN + UE containers =="',
        'ssh "$NODE" "sudo docker exec $DN_C mkdir -p $DN_RUN"',
        'scp /tmp/dn_dl_tx.mgn "$NODE:/tmp/dn_dl_tx.mgn"',
        'scp "$LOCAL/dn_ul_rx.mgn" "$NODE:/tmp/dn_ul_rx.mgn"',
        'ssh "$NODE" "sudo docker cp /tmp/dn_dl_tx.mgn $DN_C:$DN_RUN/ && sudo docker cp /tmp/dn_ul_rx.mgn $DN_C:$DN_RUN/"',
        'for i in "${!UES[@]}"; do',
        '  c="${UE_C[$i]}"',
        '  ssh "$NODE" "sudo docker exec $c mkdir -p $UE_RUN"',
        '  scp "$LOCAL/${DLRX[$i]}" "$NODE:/tmp/${DLRX[$i]}"',
        '  scp "$LOCAL/${ULTX[$i]}" "$NODE:/tmp/${ULTX[$i]}"',
        '  ssh "$NODE" "sudo docker cp /tmp/${DLRX[$i]} $c:$UE_RUN/ && sudo docker cp /tmp/${ULTX[$i]} $c:$UE_RUN/"',
        'done',
        "",
        'echo "== 0/6 teardown: kill stale mgen in this run containers =="',
        'ssh "$NODE" "sudo docker exec $DN_C pkill -9 mgen || true"',
        'for i in "${!UES[@]}"; do ssh "$NODE" "sudo docker exec ${UE_C[$i]} pkill -9 mgen || true"; done',
        'sleep 1',
        "",
        'echo "== 5/6 start receivers (DN + every UE) BEFORE senders =="',
        'ssh "$NODE" "sudo docker exec -d $DN_C mgen input $DN_RUN/dn_ul_rx.mgn output $DN_RUN/dn_ul_rx.log"',
        'for i in "${!UES[@]}"; do',
        '  c="${UE_C[$i]}"; ue="${UES[$i]}"',
        '  ssh "$NODE" "sudo docker exec -d $c mgen input $UE_RUN/${DLRX[$i]} output $UE_RUN/${ue}_dl_rx.log"',
        'done',
        'sleep 8',
        "",
        'echo "== 6/6 start senders (AFTER receivers) =="',
        'ssh "$NODE" "sudo docker exec -d $DN_C mgen txlog input $DN_RUN/dn_dl_tx.mgn output $DN_RUN/dn_dl_tx.log"',
        'for i in "${!UES[@]}"; do',
        '  c="${UE_C[$i]}"; ue="${UES[$i]}"',
        '  ssh "$NODE" "sudo docker exec -d $c mgen txlog input $UE_RUN/${ULTX[$i]} output $UE_RUN/${ue}_ul_tx.log"',
        'done',
        'echo "traffic running for ${DUR}s; senders stop automatically."',
        "",
        'echo "to collect after ~${DUR}s:"',
        'echo "  ssh $NODE sudo docker exec $DN_C grep -c RECV $DN_RUN/dn_ul_rx.log"',
    ]
    runner = deployment_dir / "deploy_rfsim.sh"
    runner.write_text("\n".join(L) + "\n")
    runner.chmod(0o755)

    # short guide
    guide = deployment_dir / "DEPLOYMENT_GUIDE.md"
    guide.write_text(f"""# RFsim deployment — {RUN_DIR.name}

    Profile: `{_tb.get('testbed')}`  |  UEs: {len(ue_mapping)}  |  duration: {dur_s}s

    UE IPs are **dynamic** (resolved live from `oaitun_ue1`). Nothing here is hand-edited.

    ## Run it
    ```bash
    bash {runner.relative_to(PROJECT_ROOT)}
    ```
    The runner performs, in order: resolve live PDU IPs (aborts if any tunnel is down)
    → rewrite the DN downlink script with those IPs → re-assert the data-plane route
    (the interface flag does not force egress on RFsim) → copy scripts into the DN and
    UE containers → start all receivers → start all senders. Logs land under
    `{dn_remote}` (DN) and `{ue_remote}` (UEs) inside the containers.
    """)

    print(f"  KIND=rfsim — emitted deployment package:")
    print(f"    {runner.relative_to(PROJECT_ROOT)}")
    print(f"    {guide.relative_to(PROJECT_ROOT)}")
    print(f"    updated_scripts/  ({len(list(updated_scripts_dir.glob('*.mgn')))} scripts, unmodified — IPs rewritten live)")

else:
    raise ValueError(f"Unknown deployment KIND: {KIND!r}")


  KIND=rfsim — emitted deployment package:
    traffic_profiles/run_filimo_igap_youtube_telegram_aparat_20260626_105335/deployment/deploy_rfsim.sh
    traffic_profiles/run_filimo_igap_youtube_telegram_aparat_20260626_105335/deployment/DEPLOYMENT_GUIDE.md
    updated_scripts/  (8 scripts, unmodified — IPs rewritten live)
